<a href="https://colab.research.google.com/github/Minakshi654/DocSense--CNN-based-Document-Classifier/blob/main/06_crewai_agent_team.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install crewai crewai-tools -q

print("Installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.5/111.5 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 809.7/809.7 kB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from google.colab import userdata
import os

os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

print("API key configured")

API key configured


In [3]:
with open("/content/ecommerce_sales_report.txt", "r") as f:
    report_text = f.read()

print("Report loaded, length:", len(report_text), "characters")

Report loaded, length: 4972 characters


In [4]:
from crewai import Agent, LLM

llm = LLM(model="gemini/gemini-2.5-flash", api_key=os.environ["GEMINI_API_KEY"])

researcher = Agent(
    role="Data Researcher",
    goal="Find precise, accurate facts from the provided business report to answer questions",
    backstory="You are a meticulous data researcher who only states facts that are explicitly "
               "present in the source material. You never guess or estimate numbers you haven't "
               "directly seen in the report.",
    llm=llm,
    verbose=True
)

print("Researcher agent created")

Researcher agent created


In [5]:
analyst = Agent(
    role="Business Analyst",
    goal="Interpret raw facts and identify meaningful business implications, trends, and risks",
    backstory="You are an experienced business analyst who excels at connecting data points to "
               "explain what they mean for the business, not just repeating numbers. You think "
               "critically about whether a figure is genuinely meaningful or potentially misleading.",
    llm=llm,
    verbose=True
)

writer = Agent(
    role="Report Writer",
    goal="Produce a clear, polished, executive-ready written answer based on the research and analysis",
    backstory="You are a skilled business writer who turns technical findings into concise, "
               "professional language suitable for a stakeholder who has limited time to read. "
               "You avoid jargon and get straight to the point.",
    llm=llm,
    verbose=True
)

print("All 3 agents created")

All 3 agents created


In [6]:
from crewai import Task

research_task = Task(
    description=f"Using ONLY the following report, extract every fact relevant to this question: "
                 f"'{{question}}'. List the specific facts and figures you find, citing the relevant "
                 f"section of the report.\n\nREPORT:\n{report_text}",
    expected_output="A bullet-point list of specific facts and figures relevant to the question, "
                     "each with a brief note on where in the report it came from.",
    agent=researcher
)

analysis_task = Task(
    description="Using the facts gathered by the researcher, analyze what they mean for the business. "
                "Identify any important caveats, risks, or nuances (e.g., is a number potentially "
                "misleading, like an average skewed by outliers?). Do not introduce new facts not "
                "provided by the researcher.",
    expected_output="A short analytical interpretation explaining the business meaning and any "
                     "important caveats of the facts provided.",
    agent=analyst,
    context=[research_task]   # this task receives the researcher's output as input
)

writing_task = Task(
    description="Using the research and analysis provided, write a final, polished answer to the "
                "original question: '{question}'. Keep it concise (3-5 sentences), professional, "
                "and easy for a busy stakeholder to read quickly.",
    expected_output="A polished, 3-5 sentence executive-ready answer to the original question.",
    agent=writer,
    context=[research_task, analysis_task]   # writer sees both prior outputs
)

print("All 3 tasks defined")

All 3 tasks defined


In [8]:
import asyncio

question = "How reliable is the average order value figure, and what should stakeholders know about it?"

result = await crew.kickoff_async(inputs={"question": question})

print("\n\n=== FINAL ANSWER ===")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0c26d768-e902-41e1-be27-4c370f2cfbb9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Using ONLY the following report, extract every fact relevant to this question: 'How reliable is the      │
│  average order value figure, and what should stakeholders know about it?'. List the specific facts and figures  │
│  you find, citing the relevant section of the report.                                                           │
│                                                                                                                 │
│  REPORT:                                                                                                        │
│  UK ONLINE RETAIL — SALES PERFORMANCE REPORT                                                                    │
│  Reporting Period: December 2010 to December 2011                                                               │
│  Prepared by: Data Science & Analytics Team                                                                     │
│                                                                                                                 │
│  EXECUTIVE SUMMARY                                                                                              │
│                                                                                                                 │
│  This report analyzes transactional sales data from a UK-based online retailer covering December 2010 through   │
│  December 2011. After data cleaning (removing 135,080 transactions with missing customer identifiers and        │
│  10,624 cancelled orders), the analysis is based on 397,884 valid transactions. Total revenue across all        │
│  markets exceeded £8.3 million during the reporting period.                                                     │
│                                                                                                                 │
│  ORDER VALUE ANALYSIS                                                                                           │
│                                                                                                                 │
│  The average order value across all transactions was £480.87. However, the median order value was               │
│  significantly lower at £303.04, indicating a right-skewed distribution driven by a small number of very large  │
│  bulk orders. The largest single order recorded was £168,469.60. This gap between average and median order      │
│  value suggests that headline "average order value" figures should be interpreted with caution, since they can  │
│  be heavily influenced by a small number of high-volume business customers rather than typical consumer         │
│  behavior. For planning purposes, the median order value of approximately £300 is a more representative figure  │
│  of a typical customer transaction.                                                                             │
│                                                                                                                 │
│  MONTHLY REVENUE TREND                                                                                          │
│                                                                                                                 │
│  Monthly revenue showed a clear upward trend across 2011, with notable seasonal acceleration in the final       │
│  quarter:                                                                                                       │
│                                                                                                                 │
│  - December 2010: £572,713.89                          

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Researcher                                                                                         │
│                                                                                                                 │
│  Task: Using ONLY the following report, extract every fact relevant to this question: 'How reliable is the      │
│  average order value figure, and what should stakeholders know about it?'. List the specific facts and figures  │
│  you find, citing the relevant section of the report.                                                           │
│                                                                                                                 │
│  REPORT:                                                                                                        │
│  UK ONLINE RETAIL — SALES PERFORMANCE REPORT                                                                    │
│  Reporting Period: December 2010 to December 2011                                                               │
│  Prepared by: Data Science & Analytics Team                                                                     │
│                                                                                                                 │
│  EXECUTIVE SUMMARY                                                                                              │
│                                                                                                                 │
│  This report analyzes transactional sales data from a UK-based online retailer covering December 2010 through   │
│  December 2011. After data cleaning (removing 135,080 transactions with missing customer identifiers and        │
│  10,624 cancelled orders), the analysis is based on 397,884 valid transactions. Total revenue across all        │
│  markets exceeded £8.3 million during the reporting period.                                                     │
│                                                                                                                 │
│  ORDER VALUE ANALYSIS                                                                                           │
│                                                                                                                 │
│  The average order value across all transactions was £480.87. However, the median order value was               │
│  significantly lower at £303.04, indicating a right-skewed distribution driven by a small number of very large  │
│  bulk orders. The largest single order recorded was £168,469.60. This gap between average and median order      │
│  value suggests that headline "average order value" figures should be interpreted with caution, since they can  │
│  be heavily influenced by a small number of high-volume business customers rather than typical consumer         │
│  behavior. For planning purposes, the median order value of approximately £300 is a more representative figure  │
│  of a typical customer transaction.                                                                             │
│                                                                                                                 │
│  MONTHLY REVENUE TREND                                                                                          │
│                                                                                                                 │
│  Monthly revenue showed a clear upward trend across 2011, with notable seasonal acceleration in the final       │
│  quarter:                                                                                                       │
│                                                        

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Researcher                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here are the facts and figures from the report relevant to the reliability of the average order value figure   │
│  and what stakeholders should know about it:                                                                    │
│                                                                                                                 │
│  *   The analysis for the report is based on 397,884 valid transactions, after cleaning 135,080 transactions    │
│  with missing customer identifiers and 10,624 cancelled orders. (Executive Summary, Data Quality Notes)         │
│  *   The average order value across all transactions was £480.87. (Order Value Analysis, Recommendations)       │
│  *   The median order value was significantly lower at £303.04. (Order Value Analysis, Recommendations)         │
│  *   This difference indicates a right-skewed distribution of order values. (Order Value Analysis)              │
│  *   The skew is driven by a small number of very large bulk orders. (Order Value Analysis, Recommendations)    │
│  *   The largest single order recorded was £168,469.60. (Order Value Analysis)                                  │
│  *   One top-performing product, "PAPER CRAFT, LITTLE BIRDIE," generated £168,469.60, primarily driven by a     │
│  single large bulk order. (Top Performing Products By Revenue)                                                  │
│  *   The gap between average and median order value suggests that headline "average order value" figures        │
│  should be interpreted with caution. (Order Value Analysis)                                                     │
│  *   The average order value can be heavily influenced by a small number of high-volume business customers      │
│  rather than typical consumer behavior. (Order Value Analysis)                                                  │
│  *   For planning purposes, the median order value of approximately £300 is a more representative figure of a   │
│  typical customer transaction. (Order Value Analysis)                                                           │
│  *   A recommendation is to use the median order value (£303.04) rather than the mean order value (£480.87)     │
│  when setting typical customer expectations, given the skew introduced by bulk orders. (Recommendations)        │
│  *   The analysis for order values reflects genuine completed sales, as it excluded transactions with missing   │
│  customer identifiers (135,080) and cancelled orders (10,624). (Data Quality Notes)                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Task: Using the research and analysis provided, write a final, polished answer to the original question: 'How  │
│  reliable is the average order value figure, and what should stakeholders know about it?'. Keep it concise      │
│  (3-5 sentences), professional, and easy for a busy stakeholder to read quickly.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The average order value of £480.87 is not fully reliable for understanding typical customer behavior, as it    │
│  is significantly inflated by a small number of very large bulk orders. Stakeholders should know that this      │
│  average is heavily influenced by high-volume business customers, not typical consumer transactions. For a      │
│  more representative understanding of what a typical customer spends, the median order value of £303.04 is a    │
│  far more accurate metric. Therefore, using the median for planning and setting customer expectations will      │
│  provide a more realistic view of sales performance.                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the research and analysis provided, write a final, polished answer to the original question: 'How  │
│  reliable is the average order value figure, and what should stakeholders know about it?'. Keep it concise      │
│  (3-5 sentences), professional, and easy for a busy stakeholder to read quickly.                                │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



=== FINAL ANSWER ===
The average order value of £480.87 is not fully reliable for understanding typical customer behavior, as it is significantly inflated by a small number of very large bulk orders. Stakeholders should know that this average is heavily influenced by high-volume business customers, not typical consumer transactions. For a more representative understanding of what a typical customer spends, the median order value of £303.04 is a far more accurate metric. Therefore, using the median for planning and setting customer expectations will provide a more realistic view of sales performance.


╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions wi

In [9]:
question2 = "Should the business invest more in international markets next year? What does the data suggest?"

result2 = await crew.kickoff_async(inputs={"question": question2})

print("\n\n=== FINAL ANSWER ===")
print(result2)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0c26d768-e902-41e1-be27-4c370f2cfbb9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Using ONLY the following report, extract every fact relevant to this question: 'Should the business      │
│  invest more in international markets next year? What does the data suggest?'. List the specific facts and      │
│  figures you find, citing the relevant section of the report.                                                   │
│                                                                                                                 │
│  REPORT:                                                                                                        │
│  UK ONLINE RETAIL — SALES PERFORMANCE REPORT                                                                    │
│  Reporting Period: December 2010 to December 2011                                                               │
│  Prepared by: Data Science & Analytics Team                                                                     │
│                                                                                                                 │
│  EXECUTIVE SUMMARY                                                                                              │
│                                                                                                                 │
│  This report analyzes transactional sales data from a UK-based online retailer covering December 2010 through   │
│  December 2011. After data cleaning (removing 135,080 transactions with missing customer identifiers and        │
│  10,624 cancelled orders), the analysis is based on 397,884 valid transactions. Total revenue across all        │
│  markets exceeded £8.3 million during the reporting period.                                                     │
│                                                                                                                 │
│  ORDER VALUE ANALYSIS                                                                                           │
│                                                                                                                 │
│  The average order value across all transactions was £480.87. However, the median order value was               │
│  significantly lower at £303.04, indicating a right-skewed distribution driven by a small number of very large  │
│  bulk orders. The largest single order recorded was £168,469.60. This gap between average and median order      │
│  value suggests that headline "average order value" figures should be interpreted with caution, since they can  │
│  be heavily influenced by a small number of high-volume business customers rather than typical consumer         │
│  behavior. For planning purposes, the median order value of approximately £300 is a more representative figure  │
│  of a typical customer transaction.                                                                             │
│                                                                                                                 │
│  MONTHLY REVENUE TREND                                                                                          │
│                                                                                                                 │
│  Monthly revenue showed a clear upward trend across 2011, with notable seasonal acceleration in the final       │
│  quarter:                                                                                                       │
│                                                                                                                 │
│  - December 2010: £572,713.89                          

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Researcher                                                                                         │
│                                                                                                                 │
│  Task: Using ONLY the following report, extract every fact relevant to this question: 'Should the business      │
│  invest more in international markets next year? What does the data suggest?'. List the specific facts and      │
│  figures you find, citing the relevant section of the report.                                                   │
│                                                                                                                 │
│  REPORT:                                                                                                        │
│  UK ONLINE RETAIL — SALES PERFORMANCE REPORT                                                                    │
│  Reporting Period: December 2010 to December 2011                                                               │
│  Prepared by: Data Science & Analytics Team                                                                     │
│                                                                                                                 │
│  EXECUTIVE SUMMARY                                                                                              │
│                                                                                                                 │
│  This report analyzes transactional sales data from a UK-based online retailer covering December 2010 through   │
│  December 2011. After data cleaning (removing 135,080 transactions with missing customer identifiers and        │
│  10,624 cancelled orders), the analysis is based on 397,884 valid transactions. Total revenue across all        │
│  markets exceeded £8.3 million during the reporting period.                                                     │
│                                                                                                                 │
│  ORDER VALUE ANALYSIS                                                                                           │
│                                                                                                                 │
│  The average order value across all transactions was £480.87. However, the median order value was               │
│  significantly lower at £303.04, indicating a right-skewed distribution driven by a small number of very large  │
│  bulk orders. The largest single order recorded was £168,469.60. This gap between average and median order      │
│  value suggests that headline "average order value" figures should be interpreted with caution, since they can  │
│  be heavily influenced by a small number of high-volume business customers rather than typical consumer         │
│  behavior. For planning purposes, the median order value of approximately £300 is a more representative figure  │
│  of a typical customer transaction.                                                                             │
│                                                                                                                 │
│  MONTHLY REVENUE TREND                                                                                          │
│                                                                                                                 │
│  Monthly revenue showed a clear upward trend across 2011, with notable seasonal acceleration in the final       │
│  quarter:                                                                                                       │
│                                                        

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Researcher                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here are the facts relevant to whether the business should invest more in international markets next year,     │
│  based solely on the provided report:                                                                           │
│                                                                                                                 │
│  *   Total revenue across all markets exceeded £8.3 million during the reporting period (December 2010 to       │
│  December 2011). (Executive Summary)                                                                            │
│  *   The United Kingdom is the dominant market, generating £7,308,391.55 in revenue. (REVENUE BY COUNTRY)       │
│  *   The UK's revenue is more than 25 times the revenue of the next closest market. (REVENUE BY COUNTRY)        │
│  *   The Netherlands generated £285,446.34 in revenue. (REVENUE BY COUNTRY)                                     │
│  *   EIRE (Ireland) generated £265,545.90 in revenue. (REVENUE BY COUNTRY)                                      │
│  *   Germany generated £228,867.14 in revenue. (REVENUE BY COUNTRY)                                             │
│  *   France generated £209,024.05 in revenue. (REVENUE BY COUNTRY)                                              │
│  *   Australia generated £138,521.31 in revenue. (REVENUE BY COUNTRY)                                           │
│  *   Spain generated £61,577.11 in revenue. (REVENUE BY COUNTRY)                                                │
│  *   Switzerland generated £56,443.95 in revenue. (REVENUE BY COUNTRY)                                          │
│  *   Belgium generated £41,196.34 in revenue. (REVENUE BY COUNTRY)                                              │
│  *   Sweden generated £38,378.33 in revenue. (REVENUE BY COUNTRY)                                               │
│  *   International markets represent a small fraction of total revenue. (REVENUE BY COUNTRY)                    │
│  *   The Netherlands, Ireland, Germany, and France each generated over £200,000, indicating meaningful          │
│  secondary markets that may warrant targeted growth initiatives. (REVENUE BY COUNTRY)                           │
│  *   One recommendation is to "Consider targeted marketing investment in the Netherlands, Ireland, Germany,     │
│  and France, which represent the strongest secondary markets outside the UK." (RECOMMENDATIONS)                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The data indicates the business should pursue a targeted investment strategy in specific international         │
│  markets next year. While the UK market overwhelmingly dominates revenue, the Netherlands, Ireland, Germany,    │
│  and France have emerged as meaningful secondary markets, each generating over £200,000. These four countries   │
│  show existing traction and are recommended for targeted marketing investment to capitalize on their            │
│  potential. Such strategic focus can help begin diversifying revenue and mitigate the current heavy reliance    │
│  on the UK market.                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the research and analysis provided, write a final, polished answer to the original question:       │
│  'Should the business invest more in international markets next year? What does the data suggest?'. Keep it     │
│  concise (3-5 sentences), professional, and easy for a busy stakeholder to read quickly.                        │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



=== FINAL ANSWER ===
The data indicates the business should pursue a targeted investment strategy in specific international markets next year. While the UK market overwhelmingly dominates revenue, the Netherlands, Ireland, Germany, and France have emerged as meaningful secondary markets, each generating over £200,000. These four countries show existing traction and are recommended for targeted marketing investment to capitalize on their potential. Such strategic focus can help begin diversifying revenue and mitigate the current heavy reliance on the UK market.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# Multi-Agent Business Analyst Crew (CrewAI)

A 3-agent collaborative AI system that researches, analyzes, and writes executive-ready answers to complex business questions — built on top of the same e-commerce sales report used in the RAG project, demonstrating how agentic reasoning extends beyond simple retrieval.

## Objective

Build a multi-agent pipeline where specialized agents each handle one stage of answering a business question — research, analysis, and communication — and demonstrate that this collaborative approach can produce recommendations not explicitly stated anywhere in the source document, going beyond what a single-shot RAG query can do.

## Architecture

Three agents, connected in a sequential pipeline using CrewAI:

1. **Data Researcher** — extracts only facts explicitly present in the source report, with an instruction to never guess or estimate. Cites the report section for each fact.
2. **Business Analyst** — receives the researcher's output as context; interprets what the facts mean, flags caveats (e.g., misleading averages), and is explicitly instructed not to introduce new facts.
3. **Report Writer** — receives both prior outputs as context; produces a concise, 3-5 sentence, stakeholder-ready final answer.

Each agent is defined with a `role`, `goal`, and `backstory`, which CrewAI combines into that agent's system prompt — the persona directly shapes output behavior, not just documentation. Task chaining is achieved via the `context` parameter, explicitly passing each task's output forward to the next.

**LLM:** Gemini 2.5 Flash, shared across all three agents.

## Test 1 — Single-section reasoning question

**Question:** "How reliable is the average order value figure, and what should stakeholders know about it?"

The Researcher correctly extracted the relevant facts (£480.87 average, £303.04 median, £168,469.60 max order) with section citations. The Analyst correctly identified the right-skewed distribution and the risk of using the average for planning. The Writer produced a polished 4-sentence answer recommending the median as the more reliable metric — matching the report's own stated recommendation, confirming the pipeline reasons correctly even on questions requiring nuanced interpretation rather than simple fact lookup.

## Test 2 — Cross-section synthesis with no explicit answer in source

**Question:** "Should the business invest more in international markets next year? What does the data suggest?"

This question has no direct answer anywhere in the source report — it required the agents to synthesize a recommendation by connecting two separate facts (UK's overwhelming dominance vs. four secondary markets each exceeding £200K) that exist in different sections of the report.

**Result:** "The data indicates the business should pursue a targeted investment strategy in specific international markets next year... These four countries show existing traction and are recommended for targeted marketing investment to capitalize on their potential. Such strategic focus can help begin diversifying revenue and mitigate the current heavy reliance on the UK market."

This demonstrates the key distinction between RAG and agentic reasoning: RAG retrieves and answers "what does the document say"; a multi-agent reasoning pipeline can synthesize "what should we do" — a genuine recommendation, balanced and proportionate to the underlying data, not explicitly written anywhere in the source.

## Technical notes / debugging

- **Async execution required in Colab:** `crew.kickoff()` raised a `RuntimeError` when called from Colab's notebook environment, since Colab runs its own event loop and CrewAI's synchronous kickoff cannot be invoked from within a running event loop. Resolved using `await crew.kickoff_async(inputs={...})`, leveraging Colab's support for top-level `await`.

## What I'd improve next

- Add a fourth "Critic" agent to review the Writer's output against the original facts before finalizing, catching any drift introduced during the analysis/writing stages
- Test with `Process.hierarchical` (a manager agent dynamically delegating tasks) instead of a fixed sequential pipeline, for more complex, open-ended questions
- Integrate the actual RAG retrieval system (ChromaDB) as a tool the Researcher agent calls, rather than passing the full report text directly — would scale to much larger documents

## Tech stack

`Python` · `CrewAI` · `Google Gemini 2.5 Flash` · Google Colab

## How to run

1. Open `06_crewai_agent_team.ipynb` in Google Colab
2. Add your Gemini API key as a Colab secret named `GEMINI_API_KEY`
3. Upload `ecommerce_sales_report.txt` (from the RAG project) to the Colab session
4. Run all cells in order